In [5]:


!pip install librosa tqdm scikit-learn tensorflow scipy pandas numpy

import os
import random
import numpy as np
import pandas as pd
import librosa
import scipy.fftpack
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# ============================================================
# PATH SETUP
# ============================================================
BASE_PATH = r"C:\Users\chaitu\Documents\sample_project_1\datafiles\freesound-audio-tagging"

TRAIN_AUDIO_PATH = os.path.join(BASE_PATH, "audio_train")
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")

# ============================================================
# LOAD & FILTER DATASET
# ============================================================
df = pd.read_csv(TRAIN_CSV)
selected_classes = df["label"].value_counts().index[:5].tolist()
df = df[df["label"].isin(selected_classes)].copy()

MAX_SAMPLES_PER_CLASS = 300
sampled_parts = [
    group.sample(n=min(len(group), MAX_SAMPLES_PER_CLASS), random_state=42)
    for _, group in df.groupby("label")
]
df = pd.concat(sampled_parts, ignore_index=True)

# ============================================================
# UPGRADED AUDIO / MFCC SETTINGS
# ============================================================
SR = 16000
DURATION = 1.0
N_MFCC = 40  # UPGRADED from 13 to 40 for higher spectral resolution
MAX_FRAMES = 99

# ============================================================
# DATA AUGMENTATION FUNCTIONS
# ============================================================
def add_noise(data, noise_factor=0.005):
    """Injects random white noise to simulate different recording environments."""
    noise = np.random.randn(len(data))
    return data + noise_factor * noise

def shift_pitch(data, sr=16000, n_steps=2):
    """Shifts pitch up or down to separate Squeaks from Laughter."""
    return librosa.effects.pitch_shift(y=data, sr=sr, n_steps=n_steps)

# ============================================================
# AUDIO PREPROCESSING
# ============================================================
def load_audio(file_path):
    y, _ = librosa.load(file_path, sr=SR, mono=True)
    return y

def pad_audio(y):
    max_len = int(SR * DURATION)
    if len(y) < max_len:
        y = np.pad(y, (0, max_len - len(y)), mode="constant")
    else:
        y = y[:max_len]
    return y

# ============================================================
# MFCC FROM SCRATCH (UPGRADED TO 48 FILTERS / 40 MFCCs)
# ============================================================
def mfcc_from_scratch(signal, sr=16000, num_mfcc=40):
    if len(signal) == 0:
        return np.zeros((1, num_mfcc))

    # 1. Pre-emphasis
    pre_emphasis = 0.97
    emphasized_signal = np.append(signal[0], signal[1:] - pre_emphasis * signal[:-1])

    # 2. Framing
    frame_size = 0.025
    frame_stride = 0.01
    frame_length = int(frame_size * sr)
    frame_step = int(frame_stride * sr)
    signal_length = len(emphasized_signal)

    num_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step)) + 1
    pad_signal_length = num_frames * frame_step + frame_length
    pad_signal = np.append(emphasized_signal, np.zeros(pad_signal_length - signal_length))

    indices = (
        np.tile(np.arange(0, frame_length), (num_frames, 1))
        + np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T
    )
    frames = pad_signal[indices.astype(np.int32, copy=False)]

    # 3. Hamming Window
    frames *= np.hamming(frame_length)

    # 4. FFT & Power Spectrum
    NFFT = 512
    mag_frames = np.absolute(np.fft.rfft(frames, NFFT))
    pow_frames = (1.0 / NFFT) * (mag_frames ** 2)

    # 5. Mel Filterbank (UPGRADED to 48 filters to support 40 MFCCs)
    nfilt = 48
    hz_to_mel = lambda hz: 2595 * np.log10(1 + hz / 700)
    mel_to_hz = lambda mel: 700 * (10 ** (mel / 2595) - 1)

    low_mel = hz_to_mel(0)
    high_mel = hz_to_mel(sr / 2)
    mel_points = np.linspace(low_mel, high_mel, nfilt + 2)
    hz_points = mel_to_hz(mel_points)
    bins = np.floor((NFFT + 1) * hz_points / sr)

    fbank = np.zeros((nfilt, int(np.floor(NFFT / 2 + 1))))
    for m in range(1, nfilt + 1):
        f_m_minus = int(bins[m - 1])
        f_m = int(bins[m])
        f_m_plus = int(bins[m + 1])

        for k in range(f_m_minus, f_m):
            if bins[m] != bins[m - 1]:
                fbank[m - 1, k] = (k - bins[m - 1]) / (bins[m] - bins[m - 1])
        for k in range(f_m, f_m_plus):
            if bins[m + 1] != bins[m]:
                fbank[m - 1, k] = (bins[m + 1] - k) / (bins[m + 1] - bins[m])

    filter_banks = np.dot(pow_frames, fbank.T)
    filter_banks = np.where(filter_banks == 0, np.finfo(float).eps, filter_banks)

    # 6. Log Filterbank Energies
    log_fbank = np.log(filter_banks)

    # 7. Discrete Cosine Transform (DCT)
    mfcc = scipy.fftpack.dct(log_fbank, type=2, axis=1, norm="ortho")[:, :num_mfcc]

    # 8. Mean Normalization
    mfcc -= np.mean(mfcc, axis=0) + 1e-8
    return mfcc


def extract_features(y):
    mfcc = mfcc_from_scratch(y, sr=SR, num_mfcc=N_MFCC)
    if mfcc.shape[0] < MAX_FRAMES:
        pad_width = MAX_FRAMES - mfcc.shape[0]
        mfcc = np.pad(mfcc, ((0, pad_width), (0, 0)), mode="constant")
    else:
        mfcc = mfcc[:MAX_FRAMES, :]
    return mfcc.T[..., np.newaxis]

# ============================================================
# EXTRACT & AUGMENT DATA (1,500 -> 4,500 Samples)
# ============================================================
X, y = [], []

print(f"\nExtracting and Augmenting features (Target: ~{len(df)*3} samples)...")
for _, row in tqdm(df.iterrows(), total=len(df)):
    file_path = os.path.join(TRAIN_AUDIO_PATH, row["fname"])
    try:
        # 1. Original Audio
        audio = load_audio(file_path)
        X.append(extract_features(pad_audio(audio)))
        y.append(row["label"])

        # 2. Add White Noise
        audio_noise = add_noise(audio)
        X.append(extract_features(pad_audio(audio_noise)))
        y.append(row["label"])

        # 3. Pitch Shift (Randomly up or down)
        steps = random.choice([-2, -1, 1, 2])
        audio_pitch = shift_pitch(audio, sr=SR, n_steps=steps)
        X.append(extract_features(pad_audio(audio_pitch)))
        y.append(row["label"])
        
    except Exception as e:
        pass

X = np.array(X, dtype=np.float32)
y = np.array(y)

print(f"\nFinal X shape: {X.shape}") # Should be ~ (4500, 40, 99, 1)

# ============================================================
# ENCODING & DATA SPLIT
# ============================================================
le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

# ============================================================
# UPGRADED 5-LAYER CNN (With L2 Regularization)
# ============================================================
# Because we upgraded N_MFCC to 40, we no longer need the (1,2) 
# hack. We can safely use standard (2,2) pooling on all 5 layers.
model = Sequential([
    Input(shape=(N_MFCC, MAX_FRAMES, 1)),

    Conv2D(32, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(128, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(256, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(512, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dropout(0.5),
    # Added L2 regularization to force the model to generalize
    Dense(256, activation="relu", kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(len(le.classes_), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# CALLBACKS & TRAINING
# ============================================================
os.makedirs("models", exist_ok=True)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1),
    ModelCheckpoint("models/best_audio_cnn_mfcc_90.keras", monitor="val_loss", save_best_only=True, verbose=1),
    # Adaptive Learning Rate: Multiplies LR by 0.5 if accuracy plateaus
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks
)

# ============================================================
# EVALUATION & METRICS
# ============================================================
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
print(f"\nTarget Achieved - Validation Accuracy: {val_acc * 100:.2f}%")

y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)

print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(y_val, y_pred, target_names=le.classes_))


Extracting and Augmenting features (Target: ~4500 samples)...


100%|███████████████████████████████████████████████████████████████████████████████████████████| 1500/1500 [03:56<00:00,  6.34it/s]



Final X shape: (4500, 40, 99, 1)
Epoch 1/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 224ms/step - accuracy: 0.3987 - loss: 2.8930 
Epoch 1: val_loss improved from None to 2.69427, saving model to models/best_audio_cnn_mfcc_90.keras

Epoch 1: finished saving model to models/best_audio_cnn_mfcc_90.keras
113/113 ━━━━━━━━━━━━━━━━━━━━ 37s 251ms/step - accuracy: 0.4736 - loss: 2.3182 - val_accuracy: 0.3156 - val_loss: 2.6943 - learning_rate: 0.0010
Epoch 2/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step - accuracy: 0.5977 - loss: 1.5474 
Epoch 2: val_loss improved from 2.69427 to 2.58651, saving model to models/best_audio_cnn_mfcc_90.keras

Epoch 2: finished saving model to models/best_audio_cnn_mfcc_90.keras
113/113 ━━━━━━━━━━━━━━━━━━━━ 39s 236ms/step - accuracy: 0.6192 - loss: 1.4671 - val_accuracy: 0.3567 - val_loss: 2.5865 - learning_rate: 0.0010
Epoch 3/50
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step - accuracy: 0.6702 - loss: 1.3083 
Epoch 3: val_loss improved from 2.58651 to 2.06999, saving model